# HUẤN LUYỆN ES-NLP PIPELINE v2.0 - GUARDIAN PULSE
## Mô hình Phân loại Ý định Thiên tai 5 nhãn (PhoBERT-v2 Fine-tuning)
**Nhãn:** `RESCUE_URGENT`, `FLOOD_HAZARD`, `OBSTACLE`, `SUPPLY_NEED`, `SPAM_IRRELEVANT`
- GPU: Google Colab T4 miễn phí
- Base Model: `vinai/phobert-base-v2`
- Framework: PyTorch + HuggingFace Transformers


In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q transformers torch scikit-learn underthesea matplotlib seaborn


In [ ]:
# 2. Upload file dữ liệu dataset_v2.csv
from google.colab import files
import pandas as pd
import io

print('Hãy tải file dataset_v2.csv từ máy bạn lên Colab:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Đã nạp {len(df)} mẫu dữ liệu. Phân bố các nhãn:')
print(df['label'].value_counts())


In [ ]:
# 3. Tiền xử lý văn bản bằng Underthesea tách từ
from underthesea import word_tokenize
from tqdm.auto import tqdm

tqdm.pandas(desc='Tách từ tiếng Việt')
df['text_seg'] = df['text'].progress_apply(lambda t: word_tokenize(str(t), format='text'))
print('Ví dụ câu sau khi tách từ:', df['text_seg'].iloc[0])


In [ ]:
# 4. Chuẩn bị Label Mapping và chia tập Train / Val / Test (80/10/10)
import numpy as np
import torch
from sklearn.model_selection import train_test_split

LABELS = ['RESCUE_URGENT', 'FLOOD_HAZARD', 'OBSTACLE', 'SUPPLY_NEED', 'SPAM_IRRELEVANT']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

df['target'] = df['label'].map(label2id)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['target'])

print(f'Tập Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


In [ ]:
# 5. Khởi tạo PhoBERT Tokenizer và Dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class DisasterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_ds = DisasterDataset(train_df['text_seg'], train_df['target'], tokenizer)
val_ds = DisasterDataset(val_df['text_seg'], val_df['target'], tokenizer)
test_ds = DisasterDataset(test_df['text_seg'], test_df['target'], tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)


In [ ]:
# 6. Khởi tạo mô hình PhoBERT Classification
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị huấn luyện:', device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
).to(device)


In [ ]:
# 7. Huấn luyện (Fine-tuning) với AdamW Optimizer
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

EPOCHS = 8
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

best_val_f1 = 0.0
best_model_path = 'best_phobert_v2.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    
    # Đánh giá trên tập Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch['label'].numpy())
            
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f'Epoch {epoch}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val F1 Macro: {val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  -> Lưu checkpoint tốt nhất với Val F1: {best_val_f1:.4f}')


In [ ]:
# 8. Đánh giá trên tập Test & Vẽ Confusion Matrix
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Nạp lại checkpoint tốt nhất
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(batch['label'].numpy())

print('=== BÁO CÁO KẾT QUẢ TRÊN TẬP TEST ===')
print(classification_report(test_labels, test_preds, target_names=LABELS, digits=4))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.title('Confusion Matrix - ES-NLP v2.0')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()


In [ ]:
# 9. Xuất mô hình để đưa về máy local
import shutil

OUTPUT_DIR = 'phobert_classifier_v2'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Tải file zip về máy:')
files.download(f'{OUTPUT_DIR}.zip')
